<a href="https://colab.research.google.com/github/rksab/NLP/blob/main/Scrapper%2Bsummarizer%2Bsentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 23.7 MB/s eta 0:00:00


In [2]:
!playwright install chromium


173.7 MiB [] 0% 132.2s173.7 MiB [] 0% 34.8s173.7 MiB [] 0% 31.4s173.7 MiB [] 0% 16.3s173.7 MiB [] 0% 8.5s173.7 MiB [] 1% 7.1s173.7 MiB [] 1% 6.2s173.7 MiB [] 2% 5.8s173.7 MiB [] 2% 5.7s173.7 MiB [] 2% 5.2s173.7 MiB [] 3% 4.7s173.7 MiB [] 3% 4.4s173.7 MiB [] 4% 4.3s173.7 MiB [] 4% 4.2s173.7 MiB [] 5% 4.2s173.7 MiB [] 5% 4.4s173.7 MiB [] 5% 4.7s173.7 MiB [] 5% 4.8s173.7 MiB [] 5% 4.9s173.7 MiB [] 6% 5.0s173.7 MiB [] 6% 5.1s173.7 MiB [] 6% 5.3s173.7 MiB [] 6% 5.4s173.7 MiB [] 7% 5.1s173.7 MiB [] 8% 4.8s173.7 MiB [] 8% 4.6s173.7 MiB [] 9% 4.6s173.7 MiB [] 10% 4.5s173.7 MiB [] 10% 4.2s173.7 MiB [] 11% 4.2s173.7 MiB [] 11% 4.1s173.7 MiB [] 12% 4.1s173.7 MiB [] 12% 4.0s173.7 MiB [] 12% 4.1s173.7 MiB [] 13% 4.1s173.7 MiB [] 14% 4.1s173.7 MiB [] 15% 4.1s173.7 MiB [] 15% 3.9s173.7 MiB [] 16% 3.8s173.7 MiB [] 17% 3.8s173.7 MiB [] 17% 3.7s173.7 MiB [] 18% 3.6s173.7 MiB [] 19% 3.6s173.7 MiB [] 20% 3.5s173.7 MiB [] 20% 3.4s173.7 MiB [] 21% 3.3s173.7 MiB [] 22% 3.3s173.7 MiB [] 22% 3.2s173.7 MiB [] 2

In [3]:
import asyncio
from playwright.async_api import async_playwright
import json
from datetime import datetime

In [4]:
import asyncio
from playwright.async_api import async_playwright

async def scrape_articles(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, timeout=60000, wait_until="domcontentloaded")

        # Main headline
        title = await page.inner_text("h1")

        # Article paragraphs (if any)
        paragraphs = await page.locator("article p").all_inner_texts()

        await browser.close()
    return title, paragraphs


In [5]:
url = "https://www.bbc.com/"
title, paragraphs = await scrape_articles(url)

print("Title:", title)
print("\nContent:")
for p in paragraphs:
    print("-", p)

Title: ‘A new foe’: Conscripting women in Denmark

Content:
- Sergei Lavrov has been highly critical of Western countries and Israel in his speech at the UN General Assembly.
- The FBI Agents Association says the terminations - not yet confirmed by the bureau - violate the agents' rights.
- Sergei Lavrov has been highly critical of Western countries and Israel in his speech at the UN General Assembly. 
- The FBI Agents Association says the terminations - not yet confirmed by the bureau - violate the agents' rights.
- The president says the latest move to send troops to a major US city is "necessary" to protect immigration detention facilities.
- As the US president pushes a crackdown on left-wing groups and openly targets political foes, critics ask where he's taking America
- Israel is continuing its offensive against Hamas, after PM Benjamin Netanyahu told the UN ​​Israel “must finish the job” in Gaza.
- Follow live text commentary as the US attempt to eat into Europe's lead in the R

In [10]:
from transformers import pipeline

# Load Hugging Face pipelines with PyTorch only
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")  # smaller & faster than bart-large
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def analyze_article(title, paragraphs):
    text = " ".join(paragraphs)

    # Summarize (limit to 1024 tokens for BART)
    summary = summarizer(text, max_length=200, min_length=30, do_sample=False)[0]["summary_text"]

    # Sentiment
    sentiment = sentiment_analyzer(summary)[0]

    return {
        "title": title,
        "summary": summary,
        "sentiment": sentiment["label"],
        "confidence": round(sentiment["score"], 3),
    }

Device set to use cpu
Device set to use cpu


In [11]:
url = "https://www.bbc.com/news/articles/c5ygjv0r2myo"
title, paragraphs = await scrape_articles(url)

result = analyze_article(title, paragraphs)

print("Title:", result["title"])
print("\nSummary:", result["summary"])
print("\nSentiment:", result["sentiment"], f"(confidence {result['confidence']})")

Title: Russia has no intention of attacking EU or Nato states, foreign minister says

Summary:  Russia's foreign minister Sergei Lavrov warns of a 'decisive response' to any 'aggression' directed towards Moscow . He said threats against Russia by Western countries are becoming "increasingly common" Russia condemned the 7 October 2023 Gaza attacks by Hamas, but said there was "no justification" for the "brutal killings" of Palestinians in Gaza . He also took aim at Israel, saying there was no justification for plans to annex the West Bank .

Sentiment: NEGATIVE (confidence 0.998)
